In [1]:
import math
import os

import numpy as np
import pandas as pd
from rdkit import Chem
from rdkit.Chem import AllChem
from sklearn.metrics.pairwise import rbf_kernel
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler, StandardScaler
from tqdm import tqdm

# Gene Exp

In [2]:
gene_exp = pd.concat(
    [
        pd.read_csv("../nci_data/gene_exp_part1.csv.gz", index_col=0).T,
        pd.read_csv("../nci_data/gene_exp_part2.csv.gz", index_col=0).T,
    ],
    axis=1,
)
gene_exp.columns = list(gene_exp.columns)
gene_exp

,A1BG,A1BG-AS1,A1CF,A2M,A2M-AS1,A2ML1,A2MP1,A3GALT2,A4GALT,A4GNT,...,ZWILCH,ZWINT,ZXDA,ZXDB,ZXDC,ZYG11A,ZYG11B,ZYX,ZZEF1,ZZZ3
MCF7,1.545084,1.960664,0.045797,0.048305,0.271383,0.000000,0.000000,0.000000,1.790422,0.000000,...,3.251321,4.553231,0.524880,1.325335,2.719968,0.979052,0.955984,2.997876,1.814542,2.017416
MDA_MB_231,0.577656,0.158440,0.023368,0.033706,0.000000,0.020794,0.000000,0.093408,0.281342,0.000000,...,4.150389,4.835798,0.399802,2.148645,1.286207,0.630559,1.366784,3.193204,0.534567,4.290867
HS578T,2.127485,1.114494,0.057868,0.053825,0.000000,0.122192,0.000000,0.000000,0.437279,0.375008,...,1.569525,2.078446,0.211162,0.425895,1.637754,0.183044,1.275240,6.707759,1.183093,1.404440
BT_549,2.807536,1.545173,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,2.705958,0.000000,...,1.551910,2.247581,0.538375,0.956725,1.921501,0.578255,1.148797,5.211335,1.806192,1.725314
T47D,2.240504,1.404554,0.014713,0.020456,0.083641,0.000000,0.000000,0.205220,0.775457,0.000000,...,2.328319,4.735132,0.634419,1.509372,2.627329,0.850154,1.326106,3.807767,1.399636,1.188945
SF_268,1.904164,0.224674,0.000000,0.066023,0.000000,0.000000,0.000000,0.256671,0.413970,0.126786,...,1.203840,3.447059,0.264858,0.741443,1.187052,0.000000,1.236389,5.422912,0.959441,1.258356
SF_295,1.645900,1.625902,0.021271,0.029497,0.000000,0.000000,0.000000,0.101272,0.493029,0.000000,...,3.338683,4.622304,0.355396,1.124633,2.284713,0.047663,1.388140,7.781976,1.643824,2.399661
SF_539,1.488978,1.449873,0.018674,0.302848,0.426903,0.000000,0.000000,0.116516,0.843719,0.000000,...,2.522377,3.481312,0.798670,1.482724,2.686960,0.119173,1.572594,7.108138,1.645789,2.314656
SNB_19,1.479329,0.274825,0.061902,0.483726,0.152568,0.076873,0.000000,0.145237,2.341513,0.302242,...,1.154635,1.465687,0.813209,1.270068,2.302293,0.121911,1.727816,4.974814,1.340579,1.763526
SNB_75,0.598351,0.248695,0.010085,6.973255,2.752582,0.027438,0.000000,0.167921,2.501286,0.093693,...,2.404099,4.063770,0.576326,1.084949,2.606243,0.095408,1.763505,6.493543,1.689898,2.246688


# Drug Response

In [3]:
drug_response = pd.read_csv("../nci_data/drugAct.csv", index_col=0)
drug_response.columns = gene_exp.index
drug_response

,MCF7,MDA_MB_231,HS578T,BT_549,T47D,SF_268,SF_295,SF_539,SNB_19,SNB_75,...,PC_3,DU_145,786_0,A498,ACHN,CAKI_1,RXF_393,SN12C,TK_10,UO_31
1,-0.271314,-0.303539,-0.815183,-0.231499,1.934731,-0.357577,-0.719253,-0.380720,-1.281589,-0.175915,...,-0.403983,-0.434885,-0.519867,-1.648059,1.657273,-0.269717,0.002290,-0.390689,-0.379420,1.059515
17,-0.354110,-0.304675,-0.222024,1.483613,1.509397,0.335572,0.424922,1.140166,-0.941890,0.330808,...,1.302516,-0.941890,0.521200,-0.329639,-0.941890,0.717851,-0.239018,-0.315533,-0.761559,1.120673
89,NaN,NaN,NaN,NaN,NaN,-0.184194,-1.429903,-0.165433,-1.429903,-0.216723,...,NaN,NaN,0.368433,-0.655707,-0.144006,0.743888,-0.435620,-0.184805,-0.101929,-0.118807
185,NaN,NaN,NaN,NaN,NaN,0.539343,NaN,0.230402,-0.765829,-1.125208,...,NaN,NaN,-0.061612,NaN,NaN,1.675353,0.918722,1.391604,-0.904724,0.918522
295,-0.264586,-0.264586,-0.264586,-0.264586,-0.264586,-0.264586,-0.264586,-0.264586,-0.264586,-0.264586,...,-0.264586,-0.264586,4.822657,-0.264586,-0.264586,-0.264586,-0.264586,-0.264586,-0.264586,-0.264586
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
900911,-0.167151,-0.167151,-0.167151,-0.167151,-0.167151,-0.167151,-0.167151,-0.167151,-0.167151,-0.167151,...,-0.167151,-0.167151,-0.167151,-0.167151,-0.167151,-0.167151,1.594059,-0.167151,-0.167151,-0.167151
900922,-0.169786,-0.169786,-0.169786,-0.169786,-0.169786,-0.169786,-0.169786,-0.169786,-0.169786,-0.169786,...,-0.169786,-0.169786,-0.169786,-0.169786,-0.169786,-0.169786,-0.169786,-0.169786,-0.169786,-0.169786
900964,-0.158754,-0.158754,-0.158754,-0.158754,-0.158754,-0.158754,-0.158754,-0.158754,-0.158754,-0.158754,...,-0.158754,NaN,-0.158754,-0.158754,-0.158754,-0.158754,-0.158754,-0.158754,-0.158754,-0.158754
900974,-0.132453,-0.132453,-0.132453,NaN,-0.132453,-0.132453,-0.132453,-0.132453,-0.132453,-0.132453,...,-0.132453,-0.132453,-0.132453,-0.132453,-0.132453,-0.132453,NaN,-0.132453,-0.132453,-0.132453


In [4]:
nsc_smiles = pd.read_csv("../Figs/nsc_cid_smiles_class_name.csv", index_col=0)[
    ["NSC", "SMILES"]
]
nsc_smiles

,NSC,SMILES
0.0,1.0,CC1=CC(=O)C=CC1=O
1.0,17.0,CCCCCCCCCCCCCCCC1=C(C=CC(=C1)O)N
2.0,89.0,CN(C)CCC(=O)C1=CC=CC=C1.Cl
3.0,185.0,CC1CC(C(=O)C(C1)C(CC2CC(=O)NC(=O)C2)O)C
4.0,758187.0,CC1CC(C(=O)C(C1)C(CC2CC(=O)NC(=O)C2)O)C
...,...,...
NaN,NaN,COCCOC1=C(C=C2C(=C1)C(=NC=N2)NC3=CC=CC(=C3)C#C...
NaN,NaN,CC1CC2C(C(C3(O2)CCC4C5CC=C6CC(CCC6(C5CC4=C3C)C...
NaN,NaN,CC(C1=C(C=CC(=C1Cl)F)Cl)OC2=C(N=CC(=C2)C3=CN(N...
NaN,NaN,CC(C)S(=O)(=O)C1=CC=CC=C1NC2=NC(=NC=C2Cl)NC3=C...


In [5]:
drug_response = drug_response[drug_response.index.isin(nsc_smiles["NSC"])]
drug_response

,MCF7,MDA_MB_231,HS578T,BT_549,T47D,SF_268,SF_295,SF_539,SNB_19,SNB_75,...,PC_3,DU_145,786_0,A498,ACHN,CAKI_1,RXF_393,SN12C,TK_10,UO_31
1,-0.271314,-0.303539,-0.815183,-0.231499,1.934731,-0.357577,-0.719253,-0.380720,-1.281589,-0.175915,...,-0.403983,-0.434885,-0.519867,-1.648059,1.657273,-0.269717,0.002290,-0.390689,-0.379420,1.059515
17,-0.354110,-0.304675,-0.222024,1.483613,1.509397,0.335572,0.424922,1.140166,-0.941890,0.330808,...,1.302516,-0.941890,0.521200,-0.329639,-0.941890,0.717851,-0.239018,-0.315533,-0.761559,1.120673
89,NaN,NaN,NaN,NaN,NaN,-0.184194,-1.429903,-0.165433,-1.429903,-0.216723,...,NaN,NaN,0.368433,-0.655707,-0.144006,0.743888,-0.435620,-0.184805,-0.101929,-0.118807
185,NaN,NaN,NaN,NaN,NaN,0.539343,NaN,0.230402,-0.765829,-1.125208,...,NaN,NaN,-0.061612,NaN,NaN,1.675353,0.918722,1.391604,-0.904724,0.918522
295,-0.264586,-0.264586,-0.264586,-0.264586,-0.264586,-0.264586,-0.264586,-0.264586,-0.264586,-0.264586,...,-0.264586,-0.264586,4.822657,-0.264586,-0.264586,-0.264586,-0.264586,-0.264586,-0.264586,-0.264586
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
830912,-0.376238,-0.376238,1.010516,-0.376238,-0.376238,-0.376238,-0.376238,-0.376238,-0.376238,-0.376238,...,-0.376238,-0.376238,-0.376238,3.525174,-0.376238,NaN,0.199423,0.443532,-0.376238,-0.376238
831147,1.072608,-0.972979,0.442335,-0.012315,-0.482196,-1.133923,0.744309,0.248854,-0.165824,0.413759,...,-0.351430,-0.514584,-0.611389,0.472510,-0.823481,0.742079,0.422323,-1.046201,-2.375068,-0.500730
831270,1.851876,-0.987088,-0.356117,-0.421592,-0.611842,-0.608036,-0.403236,-0.540247,-0.689865,-0.317744,...,-0.456405,-0.949046,-0.575863,0.841165,1.539347,2.607703,-0.341227,-0.708900,-0.597473,1.611968
831271,0.439331,-0.178969,-0.121072,0.238512,0.339249,-0.395609,-0.142904,-0.070311,-1.578078,-0.067484,...,0.108510,-0.416719,0.151945,0.340800,-0.311599,0.093151,0.349071,-0.108789,-0.327627,0.522628


In [6]:
df = pd.DataFrame()
for i in drug_response.columns:
    tmp = pd.DataFrame(drug_response[i]).reset_index().dropna()
    tmp["cell"] = [i] * len(tmp)
    tmp.columns = ["Drug", "Value", "Cell"]
    tmp = tmp[["Drug", "Cell", "Value"]]
    df = pd.concat([df, tmp])
df

,Drug,Cell,Value
0,1,MCF7,-0.271314
1,17,MCF7,-0.354110
4,295,MCF7,-0.264586
5,353,MCF7,0.744838
6,384,MCF7,-0.041323
...,...,...,...
23185,830912,UO_31,-0.376238
23186,831147,UO_31,-0.500730
23187,831270,UO_31,1.611968
23188,831271,UO_31,0.522628


# Zero shot prediction

In [7]:
unique_drugs = df["Drug"].unique()
unique_cells = df["Cell"].unique()

# Split drugs and cell lines into training, validation, and test sets
train_drugs, test_val_drugs = train_test_split(
    unique_drugs, test_size=0.5, random_state=42
)
val_drugs, test_drugs = train_test_split(test_val_drugs, test_size=0.5, random_state=42)

train_cells, test_val_cells = train_test_split(
    unique_cells, test_size=0.55, random_state=42
)
val_cells, test_cells = train_test_split(test_val_cells, test_size=0.5, random_state=42)

# Split the dataset
train_df = df[(df["Drug"].isin(train_drugs)) & (df["Cell"].isin(train_cells))]
val_df = df[(df["Drug"].isin(val_drugs)) & (df["Cell"].isin(val_cells))]
test_df = df[(df["Drug"].isin(test_drugs)) & (df["Cell"].isin(test_cells))]


# Function to balance label distribution
def balance_labels(df, threshold=0.5):
    positive = df[df["Value"] > 0]
    negative = df[df["Value"] <= 0]
    min_count = min(len(positive), len(negative))
    balanced_positive = positive.sample(min_count, random_state=42)
    balanced_negative = negative.sample(min_count, random_state=42)
    return pd.concat([balanced_positive, balanced_negative])


# Balance label distribution across all sets
train_df = balance_labels(train_df)
val_df = balance_labels(val_df)
test_df = balance_labels(test_df)

# Separate features and labels
X_train = train_df[["Drug", "Cell"]]
y_train = np.array(train_df["Value"] > 0, dtype=float)

X_val = val_df[["Drug", "Cell"]]
y_val = np.array(val_df["Value"] > 0, dtype=float)

X_test = test_df[["Drug", "Cell"]]
y_test = np.array(test_df["Value"] > 0, dtype=float)

# Calculate total samples
total_samples = len(X_train) + len(X_val) + len(X_test)


# Function to calculate and format label ratios
def get_label_ratio(y):
    unique, counts = np.unique(y, return_counts=True)
    total = len(y)
    ratio_str = ", ".join(
        [f"Label {label}: {count/total:.2%}" for label, count in zip(unique, counts)]
    )
    return ratio_str


# Display results with percentages and label ratios
print("Train:")
print(X_train.shape, y_train.shape)
print(f"Percentage: {len(X_train)/total_samples:.2%}")
print(f"Label Ratio: {get_label_ratio(y_train)}")

print("\nValidation:")
print(X_val.shape, y_val.shape)
print(f"Percentage: {len(X_val)/total_samples:.2%}")
print(f"Label Ratio: {get_label_ratio(y_val)}")

print("\nTest:")
print(X_test.shape, y_test.shape)
print(f"Percentage: {len(X_test)/total_samples:.2%}")
print(f"Label Ratio: {get_label_ratio(y_test)}")

print(f"\nTotal samples: {total_samples}")
print(
    f"Ratio (Train:Validation:Test): {len(X_train):.0f}:{len(X_val):.0f}:{len(X_test):.0f}"
)
print(
    f"Overall Label Ratio: {get_label_ratio(np.concatenate([y_train, y_val, y_test]))}"
)

X_train.to_csv("../nci_data/train.csv", index=False)
X_test.to_csv("../nci_data/test.csv", index=False)
X_val.to_csv("../nci_data/val.csv", index=False)

np.save("../nci_data/train_labels.npy", y_train)
np.save("../nci_data/test_labels.npy", y_test)
np.save("../nci_data/val_labels.npy", y_val)

Train:
(268090, 2) (268090,)
Percentage: 65.71%
Label Ratio: Label 0.0: 50.00%, Label 1.0: 50.00%

Validation:
(59402, 2) (59402,)
Percentage: 14.56%
Label Ratio: Label 0.0: 50.00%, Label 1.0: 50.00%

Test:
(80480, 2) (80480,)
Percentage: 19.73%
Label Ratio: Label 0.0: 50.00%, Label 1.0: 50.00%

Total samples: 407972
Ratio (Train:Validation:Test): 268090:59402:80480
Overall Label Ratio: Label 0.0: 50.00%, Label 1.0: 50.00%


In [8]:
drug_response = drug_response.fillna(0)
drug_response

,MCF7,MDA_MB_231,HS578T,BT_549,T47D,SF_268,SF_295,SF_539,SNB_19,SNB_75,...,PC_3,DU_145,786_0,A498,ACHN,CAKI_1,RXF_393,SN12C,TK_10,UO_31
1,-0.271314,-0.303539,-0.815183,-0.231499,1.934731,-0.357577,-0.719253,-0.380720,-1.281589,-0.175915,...,-0.403983,-0.434885,-0.519867,-1.648059,1.657273,-0.269717,0.002290,-0.390689,-0.379420,1.059515
17,-0.354110,-0.304675,-0.222024,1.483613,1.509397,0.335572,0.424922,1.140166,-0.941890,0.330808,...,1.302516,-0.941890,0.521200,-0.329639,-0.941890,0.717851,-0.239018,-0.315533,-0.761559,1.120673
89,0.000000,0.000000,0.000000,0.000000,0.000000,-0.184194,-1.429903,-0.165433,-1.429903,-0.216723,...,0.000000,0.000000,0.368433,-0.655707,-0.144006,0.743888,-0.435620,-0.184805,-0.101929,-0.118807
185,0.000000,0.000000,0.000000,0.000000,0.000000,0.539343,0.000000,0.230402,-0.765829,-1.125208,...,0.000000,0.000000,-0.061612,0.000000,0.000000,1.675353,0.918722,1.391604,-0.904724,0.918522
295,-0.264586,-0.264586,-0.264586,-0.264586,-0.264586,-0.264586,-0.264586,-0.264586,-0.264586,-0.264586,...,-0.264586,-0.264586,4.822657,-0.264586,-0.264586,-0.264586,-0.264586,-0.264586,-0.264586,-0.264586
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
830912,-0.376238,-0.376238,1.010516,-0.376238,-0.376238,-0.376238,-0.376238,-0.376238,-0.376238,-0.376238,...,-0.376238,-0.376238,-0.376238,3.525174,-0.376238,0.000000,0.199423,0.443532,-0.376238,-0.376238
831147,1.072608,-0.972979,0.442335,-0.012315,-0.482196,-1.133923,0.744309,0.248854,-0.165824,0.413759,...,-0.351430,-0.514584,-0.611389,0.472510,-0.823481,0.742079,0.422323,-1.046201,-2.375068,-0.500730
831270,1.851876,-0.987088,-0.356117,-0.421592,-0.611842,-0.608036,-0.403236,-0.540247,-0.689865,-0.317744,...,-0.456405,-0.949046,-0.575863,0.841165,1.539347,2.607703,-0.341227,-0.708900,-0.597473,1.611968
831271,0.439331,-0.178969,-0.121072,0.238512,0.339249,-0.395609,-0.142904,-0.070311,-1.578078,-0.067484,...,0.108510,-0.416719,0.151945,0.340800,-0.311599,0.093151,0.349071,-0.108789,-0.327627,0.522628


# Masking

In [9]:
# Validation Mask
for _, row in X_val.iterrows():
    drug_response.loc[row["Drug"], row["Cell"]] = 0

# Test Mask
for _, row in X_test.iterrows():
    drug_response.loc[row["Drug"], row["Cell"]] = 0

# DTI

In [10]:
dti = pd.read_csv("../data/full_table.csv")
dti = dti.dropna(subset="NSC").reset_index(drop=True)
dti.head()

,Drug Name,DrugBank ID,PubChem CID,PubChem SID,SMILES,Gene,NSC
0,Vitamin A,DB00162,445354.0,46508191.0,C\C(=C/CO)\C=C\C=C(/C)\C=C\C1=C(C)CCCC1(C)C,APOD,122759.0
1,Vitamin A,DB00162,445354.0,46508191.0,C\C(=C/CO)\C=C\C=C(/C)\C=C\C1=C(C)CCCC1(C)C,PTGDS,122759.0
2,Vitamin A,DB00162,445354.0,46508191.0,C\C(=C/CO)\C=C\C=C(/C)\C=C\C1=C(C)CCCC1(C)C,RDH12,122759.0
3,Vitamin A,DB00162,445354.0,46508191.0,C\C(=C/CO)\C=C\C=C(/C)\C=C\C1=C(C)CCCC1(C)C,RDH5,122759.0
4,Vitamin A,DB00162,445354.0,46508191.0,C\C(=C/CO)\C=C\C=C(/C)\C=C\C1=C(C)CCCC1(C)C,RDH13,122759.0


In [11]:
print("unique drugs:", len(set(dti.NSC)))
print("unique genes:", len(set(dti.Gene)))

unique drugs: 392
unique genes: 689


In [12]:
len(set(drug_response.index) & set(int(i) for i in set(dti.NSC)))

392

In [13]:
len(set(gene_exp.columns) & set(dti.Gene))

633

## All drugs are in drug response.

In [14]:
dti = dti[dti.Gene.isin(set(gene_exp.columns) & set(dti.Gene))]
dti

,Drug Name,DrugBank ID,PubChem CID,PubChem SID,SMILES,Gene,NSC
0,Vitamin A,DB00162,445354.0,46508191.0,C\C(=C/CO)\C=C\C=C(/C)\C=C\C1=C(C)CCCC1(C)C,APOD,122759.0
1,Vitamin A,DB00162,445354.0,46508191.0,C\C(=C/CO)\C=C\C=C(/C)\C=C\C1=C(C)CCCC1(C)C,PTGDS,122759.0
2,Vitamin A,DB00162,445354.0,46508191.0,C\C(=C/CO)\C=C\C=C(/C)\C=C\C1=C(C)CCCC1(C)C,RDH12,122759.0
3,Vitamin A,DB00162,445354.0,46508191.0,C\C(=C/CO)\C=C\C=C(/C)\C=C\C1=C(C)CCCC1(C)C,RDH5,122759.0
4,Vitamin A,DB00162,445354.0,46508191.0,C\C(=C/CO)\C=C\C=C(/C)\C=C\C1=C(C)CCCC1(C)C,RDH13,122759.0
...,...,...,...,...,...,...,...
1509,Trilaciclib,DB15442,NaN,NaN,CN1CCN(CC1)C1=CN=C(NC2=NC3=C(C=C4N3C3(CCCCC3)C...,CDK7,816987.0
1511,Rocaglamide,DB15495,NaN,NaN,COC1=CC=C(C=C1)[C@@]12OC3=C(C(OC)=CC(OC)=C3)[C...,PHB2,326408.0
1512,BOS172722,DB15498,NaN,NaN,CCOC1=CC(=CC=C1NC1=NC=C2C=C(C)N=C(NCC(C)(C)C)C...,TTK,809694.0
1513,Sotorasib,DB15569,NaN,NaN,CC(C)C1=NC=CC(C)=C1N1C(=O)N=C(N2CCN(C[C@@H]2C)...,KRAS,818433.0


# Selected highly variable genes

In [15]:
print(
    "Density: ",
    round((len(gene_exp.values.nonzero()[0]) / gene_exp.size) * 100, 2),
    "%",
)

Density:  75.09 %


In [16]:
variance = gene_exp.std()
variance = variance.sort_values(ascending=False)
variance = pd.DataFrame(variance > np.percentile(variance, 90))
variance = list(variance[variance[0]].index)
len(variance)

2383

In [17]:
print("DTI unique genes: ", len(set(dti["Gene"])))
print("Top 90% variable genes: ", len(variance))
print("Total: ", len(set(variance) | (set(dti["Gene"]))))

DTI unique genes:  633
Top 90% variable genes:  2383
Total:  2909


# Preprocessed data dims

In [18]:
genes = sorted(list(set(variance) | (set(dti["Gene"]))))
gene_exp = gene_exp[genes]
gene_exp.shape

(60, 2909)

In [19]:
drug_response.shape

(23190, 60)

# Normalize

In [20]:
gene_norm_cell = pd.DataFrame(
    StandardScaler().fit_transform(gene_exp),
    index=gene_exp.index,
    columns=gene_exp.columns,
)
gene_norm_cell

,A2M,AAK1,ABCA1,ABCA3,ABCB1,ABCB5,ABCC1,ABCC3,ABCC5,ABCD1,...,ZNF358,ZNF462,ZNF542P,ZNF655,ZNF667-AS1,ZNF703,ZNRD1,ZP3,ZSCAN18,ZYX
MCF7,-0.376324,-1.255054,-1.040881,2.310284,-0.370815,-0.397579,-0.579935,-0.932530,1.106057,0.387136,...,-0.285390,-0.715825,-0.917886,-2.518276,-0.779108,0.932905,0.615674,1.805885,-0.815677,-1.575934
MDA_MB_231,-0.387264,0.592792,-0.149115,-1.011604,-0.431717,-0.397579,-0.934955,-0.243114,0.676143,-1.412604,...,-1.838190,-0.228654,0.227394,1.423209,-0.779108,-0.855846,1.578712,-0.832061,1.862555,-1.454818
HS578T,-0.372187,-0.118950,-0.518109,-0.980388,-0.325888,-0.225341,-1.300565,-0.591523,-0.370487,-0.159464,...,0.802006,-1.185070,-0.243143,-0.324218,-0.114256,0.904508,-1.160363,-0.287149,-0.336022,0.724444
BT_549,-0.412522,-0.901476,-0.972737,1.331116,-0.431717,-0.397579,-1.021722,-0.142835,-1.038877,0.432084,...,1.320986,-0.119063,-0.023918,-0.986399,1.449155,-0.673092,-1.450610,1.423024,2.390315,-0.203440
T47D,-0.397193,-0.905715,-0.885907,1.878022,-0.416104,-0.397579,-1.257820,-0.091678,2.024752,-0.322621,...,-0.332664,0.400231,0.451186,0.404201,1.552933,1.229210,1.145791,0.907319,1.683829,-1.073747
SF_268,-0.363046,-0.257498,-0.904734,0.115596,-0.431717,-0.397579,-1.719511,-1.131959,-1.968854,-0.957320,...,0.438404,-0.698035,0.812874,-0.098882,1.110562,-0.351052,-0.947517,-1.084631,-0.700465,-0.072248
SF_295,-0.390418,2.217406,0.718265,-0.249157,0.686757,-0.337212,1.219555,1.348892,-0.603617,0.719695,...,1.055815,0.292586,-0.003184,0.989731,1.033240,0.322160,0.771929,2.211958,-0.882771,1.390531
SF_539,-0.185578,1.221534,-0.722066,-1.011043,-0.431717,-0.365347,0.963972,0.293545,-0.435006,1.305936,...,0.520796,0.151541,-0.239729,-0.093653,0.197136,1.737417,0.314128,0.218988,-0.544703,0.972705
SNB_19,-0.050035,-0.445755,1.098620,0.419132,-0.369670,-0.297337,-0.980557,1.303514,-0.520841,0.581188,...,2.691975,1.428340,-0.207826,0.057873,1.589442,0.204382,-1.600902,-1.091802,-0.757313,-0.350099
SNB_75,4.812989,0.974845,1.155064,-0.390628,-0.411499,-0.210385,0.746291,0.227105,0.296437,0.528289,...,1.572359,1.134967,0.427799,-0.089610,1.824346,0.272590,-0.699092,-0.740624,1.296633,0.591615


In [21]:
gene_norm_gene = pd.DataFrame(
    StandardScaler().fit_transform(gene_exp.T),
    index=gene_exp.columns,
    columns=gene_exp.index,
).T
gene_norm_gene

,A2M,AAK1,ABCA1,ABCA3,ABCB1,ABCB5,ABCC1,ABCC3,ABCC5,ABCD1,...,ZNF358,ZNF462,ZNF542P,ZNF655,ZNF667-AS1,ZNF703,ZNRD1,ZP3,ZSCAN18,ZYX
MCF7,-0.973348,-0.651963,-0.994606,0.617540,-0.966142,-0.994606,0.004154,-0.854101,0.176486,-0.028519,...,0.105497,-0.377841,-0.994606,-0.994606,-0.994606,0.592094,0.522837,0.785599,-0.917135,0.324720
MDA_MB_231,-0.887793,-0.244203,-0.619822,-0.860021,-0.900644,-0.900644,-0.120377,-0.358131,-0.002288,-0.840927,...,-0.676132,-0.168239,-0.400598,0.816960,-0.900644,-0.406198,0.791062,-0.538848,0.338803,0.316791
HS578T,-1.086677,-0.451257,-0.902592,-1.044719,-1.058152,-1.020857,-0.227938,-0.691512,-0.326631,-0.345279,...,0.756310,-0.674060,-0.736551,0.108506,-0.703985,0.625244,-0.321979,-0.339221,-0.758943,2.154055
BT_549,-1.104852,-0.668542,-1.079246,0.084745,-1.104852,-1.104852,-0.198582,-0.384464,-0.585968,-0.083005,...,0.931689,-0.177737,-0.639103,-0.308291,0.175272,-0.407552,-0.501691,0.530988,0.649778,1.265975
T47D,-1.075021,-0.678674,-1.029501,0.270902,-1.076661,-1.083679,-0.303272,-0.378803,0.318246,-0.494825,...,-0.050287,0.013266,-0.420085,0.330136,0.162751,0.604171,0.606157,0.182037,0.205433,0.527959
SF_268,-0.954411,-0.392243,-0.933047,-0.380434,-0.984791,-0.984791,-0.269753,-0.984791,-0.763314,-0.675424,...,0.575276,-0.331196,-0.072774,0.287653,0.113224,-0.088277,-0.136974,-0.684547,-0.842924,1.510495
SF_295,-1.208667,-0.178934,-0.627346,-0.843826,-0.735628,-1.193665,0.167477,0.401194,-0.628932,-0.170405,...,0.474138,-0.209199,-0.792934,0.416726,-0.286089,-0.070235,0.252928,0.625854,-1.180281,1.957261
SF_539,-1.043657,-0.272126,-1.061181,-1.130437,-1.177953,-1.162271,0.258286,-0.165970,-0.482352,0.257015,...,0.370480,-0.146281,-0.833564,0.050980,-0.631273,0.881280,0.213728,-0.210103,-0.961931,1.974115
SNB_19,-0.883731,-0.539047,-0.258857,-0.321535,-1.087165,-1.065701,-0.136760,0.779075,-0.385137,0.056579,...,1.881720,0.676454,-0.723338,0.314383,0.337013,0.178385,-0.548892,-0.805652,-1.000914,1.302890
SNB_75,1.863427,-0.403600,-0.449786,-0.917115,-1.253349,-1.170896,0.127848,-0.287505,-0.327227,-0.207083,...,0.882733,0.250314,-0.572049,-0.018399,0.211013,-0.028588,-0.322497,-0.789404,-0.096748,1.648352


# Make matrices association matrices by setting 0 threshold and min max scaler.

In [22]:
def min_max_scale(data):
    data = data[data > 0].fillna(0)
    scaler = MinMaxScaler(feature_range=(0, 1))
    return pd.DataFrame(
        scaler.fit_transform(data), index=data.index, columns=data.columns
    )

In [23]:
A_dc = min_max_scale(drug_response)
A_dc

,MCF7,MDA_MB_231,HS578T,BT_549,T47D,SF_268,SF_295,SF_539,SNB_19,SNB_75,...,PC_3,DU_145,786_0,A498,ACHN,CAKI_1,RXF_393,SN12C,TK_10,UO_31
1,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,...,0.000000,0.0,0.000000,0.000000,0.000000,0.000000,0.000303,0.000000,0.0,0.139101
17,0.000000,0.000000,0.000000,0.201118,0.205155,0.046117,0.055794,0.149733,0.0,0.043431,...,0.184168,0.0,0.072969,0.000000,0.000000,0.095068,0.000000,0.000000,0.0,0.147130
89,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,...,0.000000,0.0,0.051581,0.000000,0.000000,0.098516,0.000000,0.000000,0.0,0.000000
185,0.000000,0.000000,0.000000,0.000000,0.000000,0.074121,0.000000,0.030258,0.0,0.000000,...,0.000000,0.0,0.000000,0.000000,0.000000,0.221873,0.121670,0.202555,0.0,0.120590
295,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,...,0.000000,0.0,0.675182,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
830912,0.000000,0.000000,0.134736,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,...,0.000000,0.0,0.000000,0.470999,0.000000,0.000000,0.026410,0.000000,0.0,0.000000
831147,0.000000,0.000000,0.058978,0.000000,0.000000,0.000000,0.000000,0.032681,0.0,0.054321,...,0.000000,0.0,0.000000,0.063132,0.000000,0.098276,0.055930,0.000000,0.0,0.000000
831270,0.245251,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,...,0.000000,0.0,0.000000,0.112388,0.217647,0.345347,0.000000,0.000000,0.0,0.211631
831271,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,...,0.000000,0.0,0.021273,0.045534,0.000000,0.012336,0.046229,0.000000,0.0,0.068615


In [24]:
A_cg = min_max_scale(gene_norm_gene + gene_norm_cell)
A_cg

,A2M,AAK1,ABCA1,ABCA3,ABCB1,ABCB5,ABCC1,ABCC3,ABCC5,ABCD1,...,ZNF358,ZNF462,ZNF542P,ZNF655,ZNF667-AS1,ZNF703,ZNRD1,ZP3,ZSCAN18,ZYX
MCF7,0.000000,0.000000,0.000000,1.000000,0.000000,0.000000,0.000000,0.000000,0.378641,0.094155,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.470698,0.235570,0.766297,0.000000,0.000000
MDA_MB_231,0.000000,0.171005,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.198940,0.000000,...,0.000000,0.000000,0.000000,1.000000,0.000000,0.000000,0.490331,0.000000,0.724109,0.000000
HS578T,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.340713,0.000000,0.000000,0.000000,0.000000,0.472164,0.000000,0.000000,0.000000,0.730639
BT_549,0.000000,0.000000,0.000000,0.483588,0.000000,0.000000,0.000000,0.000000,0.000000,0.091651,...,0.492528,0.000000,0.000000,0.000000,0.376225,0.000000,0.000000,0.577798,1.000000,0.269700
T47D,0.000000,0.000000,0.000000,0.733966,0.000000,0.000000,0.000000,0.000000,0.691716,0.000000,...,0.000000,0.196455,0.006863,0.327804,0.397360,0.565881,0.362497,0.322121,0.621449,0.000000
SF_268,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.221632,0.000000,0.163323,0.084266,0.283435,0.000000,0.000000,0.000000,0.000000,0.365065
SF_295,0.000000,1.000000,0.033782,0.000000,0.000000,0.000000,0.612423,0.555747,0.000000,0.144216,...,0.334511,0.039617,0.000000,0.627835,0.173043,0.077758,0.212054,0.839136,0.000000,0.849758
SF_539,0.000000,0.465745,0.000000,0.000000,0.000000,0.000000,0.539669,0.040512,0.000000,0.410353,...,0.194870,0.002499,0.000000,0.000000,0.000000,0.808272,0.109219,0.002627,0.000000,0.747981
SNB_19,0.000000,0.000000,0.312023,0.033334,0.000000,0.000000,0.000000,0.661335,0.000000,0.167446,...,1.000000,1.000000,0.000000,0.166173,0.446176,0.118143,0.000000,0.000000,0.000000,0.241844
SNB_75,1.000000,0.280232,0.262054,0.000000,0.000000,0.000000,0.385963,0.000000,0.000000,0.084333,...,0.536785,0.658155,0.000000,0.000000,0.471398,0.075312,0.000000,0.000000,0.394687,0.568563


In [25]:
A_dg = (
    pd.DataFrame(
        np.ones([len(A_dc.index), len(A_cg.columns)]),
        index=A_dc.index,
        columns=A_cg.columns,
    )
    / 2
)
for _, i in dti.iterrows():
    A_dg.loc[int(i["NSC"]), i["Gene"]] = 1
A_dg

,A2M,AAK1,ABCA1,ABCA3,ABCB1,ABCB5,ABCC1,ABCC3,ABCC5,ABCD1,...,ZNF358,ZNF462,ZNF542P,ZNF655,ZNF667-AS1,ZNF703,ZNRD1,ZP3,ZSCAN18,ZYX
1,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,...,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5
17,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,...,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5
89,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,...,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5
185,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,...,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5
295,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,...,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
830912,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,...,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5
831147,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,...,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5
831270,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,...,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5
831271,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,...,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5


In [26]:
A_dg.sum().sum()

33730579.0

In [27]:
print(
    "Drug Density: ",
    round(len(A_dc.values.nonzero()[0]) / drug_response.size, 4) * 100,
    "%",
)
print("Cell Density: ", round(len(A_cg.values.nonzero()[0]) / A_cg.size, 4) * 100, "%")
print("Gene Density: ", round(len(A_dg.values.nonzero()[0]) / A_dg.size, 5) * 100, "%")

Drug Density:  34.339999999999996 %
Cell Density:  45.1 %
Gene Density:  100.0 %


# Similarity

In [28]:
def normalize_similarity_matrix(df, gamma=None):
    similarity_matrix = rbf_kernel(df.values, gamma=gamma)
    scaler = MinMaxScaler()
    normalized_matrix = scaler.fit_transform(similarity_matrix.reshape(-1, 1))
    normalized_df = pd.DataFrame(
        normalized_matrix.reshape(similarity_matrix.shape),
        index=df.index,
        columns=df.index,
    )

    return normalized_df

In [29]:
cell_sim = normalize_similarity_matrix(drug_response.T)
cell_sim.to_csv("../nci_data/cell_sim.csv")
cell_sim

,MCF7,MDA_MB_231,HS578T,BT_549,T47D,SF_268,SF_295,SF_539,SNB_19,SNB_75,...,PC_3,DU_145,786_0,A498,ACHN,CAKI_1,RXF_393,SN12C,TK_10,UO_31
MCF7,1.000000,0.227447,0.143721,0.256653,0.256470,0.290226,0.262940,0.272782,0.185349,0.124048,...,0.346101,0.258853,0.313270,0.099894,0.304046,0.168053,0.230009,0.305532,0.150460,0.136393
MDA_MB_231,0.227447,1.000000,0.259775,0.381369,0.255845,0.400184,0.310891,0.304846,0.320464,0.160888,...,0.440578,0.333731,0.396604,0.141942,0.358157,0.207702,0.276867,0.438735,0.288601,0.172283
HS578T,0.143721,0.259775,1.000000,0.288525,0.160353,0.294563,0.255554,0.233412,0.257183,0.169901,...,0.296448,0.233667,0.249829,0.149023,0.214025,0.127521,0.209183,0.262126,0.182485,0.106441
BT_549,0.256653,0.381369,0.288525,1.000000,0.298805,0.456849,0.351465,0.362729,0.378570,0.185974,...,0.471761,0.404830,0.430048,0.170918,0.386019,0.232237,0.309199,0.457730,0.300321,0.198983
T47D,0.256470,0.255845,0.160353,0.298805,1.000000,0.287528,0.232394,0.212602,0.214370,0.129672,...,0.336931,0.258686,0.283670,0.108305,0.284543,0.169274,0.215131,0.298715,0.212876,0.152811
SF_268,0.290226,0.400184,0.294563,0.456849,0.287528,1.000000,0.410619,0.408225,0.454986,0.204139,...,0.491892,0.480305,0.498097,0.161695,0.465027,0.249244,0.315546,0.541127,0.308117,0.207663
SF_295,0.262940,0.310891,0.255554,0.351465,0.232394,0.410619,1.000000,0.358621,0.387376,0.184207,...,0.462864,0.368207,0.386247,0.188060,0.365875,0.220585,0.264474,0.386052,0.224370,0.170811
SF_539,0.272782,0.304846,0.233412,0.362729,0.212602,0.408225,0.358621,1.000000,0.309519,0.179080,...,0.394664,0.362469,0.435349,0.138195,0.364717,0.202107,0.287604,0.393333,0.199906,0.157559
SNB_19,0.185349,0.320464,0.257183,0.378570,0.214370,0.454986,0.387376,0.309519,1.000000,0.168058,...,0.401501,0.406316,0.371649,0.159504,0.325493,0.173616,0.222217,0.435851,0.265908,0.144649
SNB_75,0.124048,0.160888,0.169901,0.185974,0.129672,0.204139,0.184207,0.179080,0.168058,1.000000,...,0.209472,0.164562,0.188763,0.093199,0.165335,0.097829,0.176725,0.183361,0.107629,0.085996


In [30]:
print("Min:", np.min(cell_sim.values))
print("Max:", np.max(cell_sim.values))
print("Mean:", np.mean(cell_sim.values))
print("Median:", np.median(cell_sim.values))

Min: 0.0
Max: 1.0
Mean: 0.2449607026243411
Median: 0.23438909231468436


In [31]:
gene_sim = normalize_similarity_matrix(gene_norm_cell.T)
gene_sim.to_csv("../nci_data/gene_sim.csv")
gene_sim

,A2M,AAK1,ABCA1,ABCA3,ABCB1,ABCB5,ABCC1,ABCC3,ABCC5,ABCD1,...,ZNF358,ZNF462,ZNF542P,ZNF655,ZNF667-AS1,ZNF703,ZNRD1,ZP3,ZSCAN18,ZYX
A2M,1.000000,0.226306,0.119409,0.060190,0.090301,0.391793,0.099959,0.062216,0.166070,0.340668,...,0.185882,0.210333,0.128977,0.165382,0.119324,0.222604,0.069420,0.076629,0.097455,0.140388
AAK1,0.226306,1.000000,0.290272,0.088265,0.107515,0.196625,0.336649,0.178529,0.161851,0.294878,...,0.339250,0.473961,0.191904,0.315834,0.127629,0.169825,0.103541,0.125219,0.158049,0.416474
ABCA1,0.119409,0.290272,1.000000,0.129157,0.128088,0.092673,0.182756,0.271109,0.098094,0.123428,...,0.186187,0.301942,0.091080,0.204786,0.112253,0.093703,0.058303,0.115773,0.140000,0.258915
ABCA3,0.060190,0.088265,0.129157,1.000000,0.275980,0.048084,0.215076,0.174623,0.159356,0.058527,...,0.194404,0.154588,0.187560,0.078862,0.244293,0.101630,0.123893,0.361435,0.203343,0.167400
ABCB1,0.090301,0.107515,0.128088,0.275980,1.000000,0.087243,0.240955,0.132595,0.166354,0.080465,...,0.176134,0.141762,0.229688,0.131027,0.180989,0.128992,0.187949,0.200042,0.174078,0.223905
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
ZNF703,0.222604,0.169825,0.093703,0.101630,0.128992,0.224850,0.157619,0.175506,0.071109,0.230201,...,0.141732,0.190102,0.101912,0.068699,0.103096,1.000000,0.058706,0.178124,0.050863,0.194664
ZNRD1,0.069420,0.103541,0.058303,0.123893,0.187949,0.086748,0.154868,0.054900,0.418163,0.054293,...,0.055761,0.058149,0.244632,0.189314,0.128546,0.058706,1.000000,0.126709,0.250182,0.083467
ZP3,0.076629,0.125219,0.115773,0.361435,0.200042,0.064751,0.230891,0.211739,0.172942,0.119484,...,0.155753,0.153724,0.125526,0.079986,0.181292,0.178124,0.126709,1.000000,0.159002,0.197240
ZSCAN18,0.097455,0.158049,0.140000,0.203343,0.174078,0.059511,0.160625,0.085757,0.245629,0.075171,...,0.188774,0.133152,0.415823,0.193341,0.380405,0.050863,0.250182,0.159002,1.000000,0.135090


In [32]:
print("Min:", np.min(gene_sim.values))
print("Max:", np.max(gene_sim.values))
print("Mean:", np.mean(gene_sim.values))
print("Median:", np.median(gene_sim.values))

Min: 0.0
Max: 1.0
Mean: 0.14968046858753645
Median: 0.12804139207924833


# NSC to SMILES

In [33]:
convert = dict(
    pd.read_csv("../Figs/nsc_cid_smiles_class_name.csv", index_col=0)[
        ["NSC", "SMILES"]
    ].values
)
SMILES = [convert[i] for i in drug_response.index]
SMILES

['CC1=CC(=O)C=CC1=O',
 'CCCCCCCCCCCCCCCC1=C(C=CC(=C1)O)N',
 'CN(C)CCC(=O)C1=CC=CC=C1.Cl',
 'CC1CC(C(=O)C(C1)C(CC2CC(=O)NC(=O)C2)O)C',
 'C1=CC=C(C=C1)CCCC(=O)O',
 'CCN(CC)CCCNC1=C2C=C(C=CC2=NC3=C1C=CC(=C3)Cl)OC.Cl',
 'CCCCCN(CCCCC)CCCNC1=C2CCCCC2=NC3=C1C=C(C=C3)Cl.OP(=O)(O)O',
 'CC1=C(C(=C(C=C1[N+](=O)[O-])[N+](=O)[O-])Br)[N+](=O)[O-]',
 'C1C(O1)C2CO2',
 'C1=CC=C2C(=C1)C(=C(N2)O)N=NC(=S)N',
 'C1=CC(=CC=C1C(=O)NC(CCC(=O)O)C(=O)O)NCC2=CN=C3C(=N2)C(=NC(=N3)N)N',
 'CN(CC1=CN=C2C(=N1)C(=NC(=N2)N)N)C3=CC=C(C=C3)C(=O)NC(CCC(=O)O)C(=O)O',
 'C(C(C(=O)O)N)OC(=O)C=[N+]=N',
 'C12=NNN=C1N=C(NC2=O)N',
 'C1=NC2=C(N1)C(=S)N=C(N2)N',
 'C1=NC2=C(N1)C(=S)N=CN2',
 'CC(=O)NC1CCC2=CC(=C(C(=C2C3=CC=C(C(=O)C=C13)OC)OC)OC)OC',
 'CN(CCCl)CCCl.Cl',
 'C1=CC=C(C=C1)C=CC(=O)C2=CC=CC=N2',
 'C1=CN=CC=C1C=CC(=O)O',
 'CN(C)C1=C(C=C(C=C1)C(=O)C2=CC(=C(C=C2)N(C)C)N)N',
 'C1=CC=C(C=C1)C(C2=C(C3=C(C=CC=N3)C=C2)O)NC4=CC=C(C=C4)[N+](=O)[O-]',
 'C1=CC=C(C=C1)C(C2=C(C3=C(C=CC=N3)C=C2)O)NC4=CC=C(C=C4)C(=O)O',
 'C1=CC=C(C=C1)C(C2

In [34]:
def get_morgan_fingerprint(SMILES):
    # Initialize parser parameters
    params = Chem.SmilesParserParams()
    params.useChirality = True  # Preserve stereochemistry information
    params.removeHs = False  # Keep hydrogen atoms
    mfp = []

    for smi in SMILES:
        mol = None
        # Sanitization attempt strategies
        sanitize_attempts = [
            {"sanitize": True},  # First try with standard sanitization
            {
                "sanitize": False,
                "partial_sanitize": True,
            },  # Fallback: partial sanitization
        ]

        for attempt in sanitize_attempts:
            try:
                # Update parameters for this attempt
                current_params = Chem.SmilesParserParams()
                current_params.sanitize = attempt["sanitize"]
                current_params.useChirality = params.useChirality
                current_params.removeHs = params.removeHs

                # Molecule object creation
                mol = Chem.MolFromSmiles(smi, params=current_params)

                if mol and "partial_sanitize" in attempt:
                    # Perform partial sanitization (skip property validation)
                    Chem.SanitizeMol(mol, Chem.SANITIZE_ALL ^ Chem.SANITIZE_PROPERTIES)

                if mol:  # Successfully processed molecule
                    # Generate Morgan fingerprint
                    fp = AllChem.GetMorganFingerprintAsBitVect(mol, 2, nBits=2048)
                    mfp.append(np.array(fp))
                    break  # Exit attempt loop on success

            except Exception as e:
                if attempt == sanitize_attempts[-1]:  # Final attempt failed
                    print(f"Processing failed: {smi}")
                    print(f"Error details: {str(e)}")
                continue  # Try next attempt

        if not mol:  # All attempts failed
            print(f"Complete processing failure: {smi}")
            mfp.append(np.zeros(2048))  # Insert zero-vector placeholder

    return np.array(mfp)

In [35]:
mfp = get_morgan_fingerprint(SMILES)
mfp = pd.DataFrame(mfp, index=drug_response.index)

[13:29:18] Explicit valence for atom # 4 Sn, 5, is greater than permitted


In [36]:
drug_sim = normalize_similarity_matrix(mfp)
drug_sim

,1,17,89,185,295,353,384,596,629,721,...,829498,830173,830201,830202,830368,830912,831147,831270,831271,831453
1,1.000000,0.846911,0.866893,0.834946,0.874899,0.767472,0.747733,0.874899,0.902982,0.826979,...,0.767472,0.638082,0.700556,0.661452,0.684892,0.700556,0.815043,0.696637,0.704477,0.638082
17,0.846911,1.000000,0.834946,0.779339,0.858894,0.767472,0.771426,0.819019,0.854898,0.811068,...,0.728042,0.614779,0.684892,0.645864,0.661452,0.661452,0.767472,0.673163,0.657552,0.607027
89,0.866893,0.834946,1.000000,0.799155,0.902982,0.779339,0.759571,0.830961,0.866893,0.822998,...,0.747733,0.673163,0.720180,0.665354,0.688805,0.673163,0.819019,0.700556,0.669258,0.641972
185,0.834946,0.779339,0.799155,1.000000,0.807095,0.692720,0.688805,0.783298,0.842921,0.759571,...,0.684892,0.603154,0.657552,0.626422,0.626422,0.641972,0.787259,0.653654,0.653654,0.595413
295,0.874899,0.858894,0.902982,0.807095,1.000000,0.771426,0.759571,0.838933,0.882913,0.838933,...,0.755623,0.657552,0.735913,0.696637,0.696637,0.688805,0.803124,0.708400,0.684892,0.657552
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
830912,0.700556,0.661452,0.673163,0.641972,0.688805,0.622539,0.603154,0.657552,0.692720,0.634193,...,0.638082,0.541437,0.572235,0.541437,0.564524,1.000000,0.653654,0.552972,0.568379,0.526083
831147,0.815043,0.767472,0.819019,0.787259,0.803124,0.696637,0.692720,0.763521,0.815043,0.747733,...,0.688805,0.630307,0.684892,0.630307,0.630307,0.653654,1.000000,0.688805,0.649758,0.622539
831270,0.696637,0.673163,0.700556,0.653654,0.708400,0.641972,0.661452,0.653654,0.696637,0.645864,...,0.579953,0.568379,0.599282,0.545280,0.545280,0.552972,0.688805,1.000000,0.572235,0.552972
831271,0.704477,0.657552,0.669258,0.653654,0.684892,0.610902,0.583815,0.677071,0.720180,0.669258,...,0.595413,0.537596,0.599282,0.552972,0.576093,0.568379,0.649758,0.572235,1.000000,0.560672


In [37]:
os.makedirs("../nci_data/drug_sim", exist_ok=True)
chunks = np.array_split(drug_sim, 25)

for i, chunk in tqdm(enumerate(chunks)):
    chunk.to_parquet(
        f"../nci_data/drug_sim/drug_sim_part_{i}.parquet", compression="gzip"
    )

/Users/inouey2/miniconda3/envs/torch/lib/python3.10/site-packages/numpy/core/fromnumeric.py:59: FutureWarning: 'DataFrame.swapaxes' is deprecated and will be removed in a future version. Please use 'DataFrame.transpose' instead.
  return bound(*args, **kwds)
25it [01:11,  2.84s/it]


In [38]:
# # How to read
# def natural_sort_key(s):
#     return [int(c) if c.isdigit() else c for c in re.split(r'(\d+)', s)]

# file_paths = glob.glob("../nci_data/drug_sim/drug_sim_part_*.parquet")
# sorted_file_paths = sorted(file_paths, key=natural_sort_key)

# drug_sim_reconstructed = pd.concat([pd.read_parquet(file) for file in tqdm(sorted_file_paths)])

In [39]:
print("Min:", np.min(drug_sim.values))
print("Max:", np.max(drug_sim.values))
print("Mean:", np.mean(drug_sim.values))
print("Median:", np.median(drug_sim.values))

Min: 0.0
Max: 1.0
Mean: 0.7012786922387917
Median: 0.708399735004468


# Unified Graph

In [40]:
A_cg

,A2M,AAK1,ABCA1,ABCA3,ABCB1,ABCB5,ABCC1,ABCC3,ABCC5,ABCD1,...,ZNF358,ZNF462,ZNF542P,ZNF655,ZNF667-AS1,ZNF703,ZNRD1,ZP3,ZSCAN18,ZYX
MCF7,0.000000,0.000000,0.000000,1.000000,0.000000,0.000000,0.000000,0.000000,0.378641,0.094155,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.470698,0.235570,0.766297,0.000000,0.000000
MDA_MB_231,0.000000,0.171005,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.198940,0.000000,...,0.000000,0.000000,0.000000,1.000000,0.000000,0.000000,0.490331,0.000000,0.724109,0.000000
HS578T,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.340713,0.000000,0.000000,0.000000,0.000000,0.472164,0.000000,0.000000,0.000000,0.730639
BT_549,0.000000,0.000000,0.000000,0.483588,0.000000,0.000000,0.000000,0.000000,0.000000,0.091651,...,0.492528,0.000000,0.000000,0.000000,0.376225,0.000000,0.000000,0.577798,1.000000,0.269700
T47D,0.000000,0.000000,0.000000,0.733966,0.000000,0.000000,0.000000,0.000000,0.691716,0.000000,...,0.000000,0.196455,0.006863,0.327804,0.397360,0.565881,0.362497,0.322121,0.621449,0.000000
SF_268,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.221632,0.000000,0.163323,0.084266,0.283435,0.000000,0.000000,0.000000,0.000000,0.365065
SF_295,0.000000,1.000000,0.033782,0.000000,0.000000,0.000000,0.612423,0.555747,0.000000,0.144216,...,0.334511,0.039617,0.000000,0.627835,0.173043,0.077758,0.212054,0.839136,0.000000,0.849758
SF_539,0.000000,0.465745,0.000000,0.000000,0.000000,0.000000,0.539669,0.040512,0.000000,0.410353,...,0.194870,0.002499,0.000000,0.000000,0.000000,0.808272,0.109219,0.002627,0.000000,0.747981
SNB_19,0.000000,0.000000,0.312023,0.033334,0.000000,0.000000,0.000000,0.661335,0.000000,0.167446,...,1.000000,1.000000,0.000000,0.166173,0.446176,0.118143,0.000000,0.000000,0.000000,0.241844
SNB_75,1.000000,0.280232,0.262054,0.000000,0.000000,0.000000,0.385963,0.000000,0.000000,0.084333,...,0.536785,0.658155,0.000000,0.000000,0.471398,0.075312,0.000000,0.000000,0.394687,0.568563


In [41]:
indexes = list(A_dc.index) + list(A_cg.index) + list(A_dg.columns)
n_all = len(indexes)
base = pd.DataFrame(np.zeros([n_all, n_all]), index=indexes, columns=indexes)
base

,1,17,89,185,295,353,384,596,629,721,...,ZNF358,ZNF462,ZNF542P,ZNF655,ZNF667-AS1,ZNF703,ZNRD1,ZP3,ZSCAN18,ZYX
1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
17,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
89,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
185,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
295,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
ZNF703,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
ZNRD1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
ZP3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
ZSCAN18,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [42]:
base.loc[A_cg.index, A_cg.columns] = A_cg
base.loc[A_cg.columns, A_cg.index] = A_cg.T
base.loc[A_dc.index, A_dc.columns] = A_dc
base.loc[A_dc.columns, A_dc.index] = A_dc.T
base.loc[A_dg.index, A_dg.columns] = A_dg
base.loc[A_dg.columns, A_dg.index] = A_dg.T

In [43]:
np.save(
    "../nci_data/idxs.npy",
    pd.DataFrame([list(range(len(base.index))), base.index]).values,
)

In [44]:
edge_index = np.array(base.values.nonzero())
edge_attr = np.array(base.values[base.values.nonzero()])

In [45]:
os.makedirs("../nci_data/edge_idxs", exist_ok=True)
chunk_size = 3000000

num_chunks_index = math.ceil(edge_index.shape[1] / chunk_size)
for i in range(num_chunks_index):
    start = i * chunk_size
    end = min((i + 1) * chunk_size, edge_index.shape[1])
    chunk = edge_index[:, start:end]
    np.save(f"../nci_data/edge_idxs/edge_idxs_chunk_{i}.npy", chunk)

In [46]:
os.makedirs("../nci_data/edge_attrs", exist_ok=True)
chunk_size = 6000000

num_chunks_attr = math.ceil(edge_attr.shape[0] / chunk_size)
for i in range(num_chunks_attr):
    start = i * chunk_size
    end = min((i + 1) * chunk_size, edge_attr.shape[0])
    chunk = edge_attr[start:end]
    np.save(f"../nci_data/edge_attrs/edge_attr_chunk_{i}.npy", chunk)

In [47]:
# # How to read
# def load_and_combine_chunks(pattern, axis=0):
#     chunk_files = sorted(
#         glob.glob(pattern), key=lambda x: int(x.split("_")[-1].split(".")[0])
#     )

#     chunks = [np.load(f) for f in chunk_files]
#     return np.concatenate(chunks, axis=axis)


# edge_index = load_and_combine_chunks("../nci_data/edge_idxs/*.npy", axis=1)
# edge_attr = load_and_combine_chunks("../nci_data/edge_attrs/*.npy", axis=0)